# Day 28 Tutorial：模型卡与课程总报告

## Goal

把统一结果表转换成包含选择规则、限制和复现信息的模型卡。

## Setup

使用确定性人工结果表，不读取个人文件，也不重新训练模型。

In [1]:
import json
import pandas as pd

results = pd.DataFrame([
    ('dummy', 'cv_valid', 1.20, 0.08, 5, 'table_v1'),
    ('ridge', 'cv_valid', 0.82, 0.05, 5, 'table_v1'),
    ('random_forest', 'cv_valid', 0.78, 0.07, 5, 'table_v1'),
    ('mlp', 'cv_valid', 0.80, 0.11, 5, 'table_v1'),
], columns=['model', 'split', 'rmse_mean', 'rmse_std', 'n_repeats', 'input_set'])
results

,model,split,rmse_mean,rmse_std,n_repeats,input_set
0,dummy,cv_valid,1.20,0.08,5,table_v1
1,ridge,cv_valid,0.82,0.05,5,table_v1
2,random_forest,cv_valid,0.78,0.07,5,table_v1
3,mlp,cv_valid,0.80,0.11,5,table_v1


## Steps

### 1. 只在相同协议下选择验证候选

In [2]:
comparable = (
    results['split'].nunique() == 1
    and results['input_set'].nunique() == 1
    and results['n_repeats'].nunique() == 1
)
selected = results.loc[results['rmse_mean'].idxmin(), 'model']
print('comparable:', comparable)
print('selected by CV RMSE:', selected)

comparable: True
selected by CV RMSE: random_forest


### 2. 形成可审计模型卡

In [3]:
model_card = {
    'task': 'regression',
    'data': {'sample_definition': 'one row per object', 'input_set': 'table_v1'},
    'split_protocol': 'five-fold cross-validation',
    'models': results['model'].tolist(),
    'metrics': ['rmse_mean', 'rmse_std'],
    'results': {'selected_by_cv': selected},
    'limitations': [
        'synthetic teaching table',
        'no external-distribution evaluation',
    ],
    'reproducibility': {'n_repeats': 5, 'test_used_for_selection': False},
}
print(json.dumps(model_card, ensure_ascii=False, indent=2))

{
  "task": "regression",
  "data": {
    "sample_definition": "one row per object",
    "input_set": "table_v1"
  },
  "split_protocol": "five-fold cross-validation",
  "models": [
    "dummy",
    "ridge",
    "random_forest",
    "mlp"
  ],
  "metrics": [
    "rmse_mean",
    "rmse_std"
  ],
  "results": {
    "selected_by_cv": "random_forest"
  },
  "limitations": [
    "synthetic teaching table",
    "no external-distribution evaluation"
  ],
  "reproducibility": {
    "n_repeats": 5,
    "test_used_for_selection": false
  }
}


## Checks

验证基线、选择规则和模型卡必需字段。

In [4]:
required = {
    'task', 'data', 'split_protocol', 'models', 'metrics',
    'results', 'limitations', 'reproducibility',
}
assert comparable
assert 'dummy' in model_card['models']
assert required.issubset(model_card)
assert model_card['reproducibility']['test_used_for_selection'] is False
assert model_card['limitations']
print('Checks passed: model card is complete and test-blind.')

Checks passed: model card is complete and test-blind.


## Next Steps

独立完成练习，并把下一步与最重要的证据缺口对应起来。